# FinBERT — FinancialPhraseBank Sentiment Pipeline

**Flow**
1. Load **FinancialPhraseBank** (75% agreement) — stratified **60/20/20** (train/val/test) split.
2. **Stage 1 – Baseline**: pretrained FinBERT on FPB **test set** (20%).
3. **Stage 2 – LoRA fine-tuning + Random Search**: train on 60%, `compute_metrics` reports val **macro-F1** each eval step, EarlyStopping on val macro-F1.
4. **Final**: retrain on train+val (80%) → evaluate on **test set** → save full model.
5. All outputs → `genai_grp_project/finbert_sentiment/`.

In [ ]:
%%capture
!pip install -U transformers peft bitsandbytes accelerate datasets scikit-learn pandas python-dotenv requests torch

In [ ]:
import subprocess, torch
try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
except FileNotFoundError:
    print("nvidia-smi not found")

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
import os, gc, io, random, json
from pathlib import Path
import numpy as np, pandas as pd, requests
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

for p in [Path(os.getcwd()), *Path(os.getcwd()).parents]:
    if (p / ".env").exists(): load_dotenv(p / ".env", override=False); print(f"Loaded .env from {p}"); break

CKPT_BASE = Path(os.getenv("CHECKPOINT_DIR", "genai_grp_project")) / "finbert_sentiment"
CKPT_BASE.mkdir(parents=True, exist_ok=True)

MODEL_NAME           = "ProsusAI/finbert"
LABELS               = ["negative", "neutral", "positive"]
LABEL_SET            = set(LABELS)
LABEL2ID             = {l: i for i, l in enumerate(LABELS)}
ID2LABEL             = {i: l for i, l in enumerate(LABELS)}
RANDOM_SEARCH_TRIALS = int(os.getenv("RANDOM_SEARCH_TRIALS", 4))

print(f"CKPT_BASE : {CKPT_BASE}\nTrials    : {RANDOM_SEARCH_TRIALS}")

In [ ]:
FPB_URL = ("https://raw.githubusercontent.com/maxwellsarpong/"
           "NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt")

resp = requests.get(FPB_URL, timeout=60); resp.raise_for_status()
fpb_df = pd.read_csv(io.StringIO(resp.text), sep="@", header=None,
                     names=["sentence","label_text"], engine="python", encoding="latin-1")
fpb_df["sentence"]   = fpb_df["sentence"].str.strip()
fpb_df["label_text"] = fpb_df["label_text"].str.strip().str.lower()
fpb_df = fpb_df[fpb_df["label_text"].isin(LABEL_SET)].reset_index(drop=True)
fpb_df["label"]      = fpb_df["label_text"].map(LABEL2ID)
print(f"FPB: {len(fpb_df)} samples"); print(fpb_df["label_text"].value_counts())

train_val_df, test_df = train_test_split(fpb_df, test_size=0.20, random_state=SEED, stratify=fpb_df["label_text"])
train_df, val_df      = train_test_split(train_val_df, test_size=0.25, random_state=SEED, stratify=train_val_df["label_text"])
print(f"\nTrain {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")

In [ ]:
# ─── Shared helpers ───────────────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from torch.utils.data import Dataset as TorchDataset

class FPBDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        enc = tokenizer(df["sentence"].tolist(), truncation=True, padding="max_length",
                        max_length=max_length, return_tensors="pt")
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(df["label"].tolist(), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "attention_mask": self.attention_mask[idx],
                "labels": self.labels[idx]}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    pred_labels = [ID2LABEL[p] for p in preds]
    true_labels = [ID2LABEL[l] for l in labels]
    return {"macro_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="macro", zero_division=0)}

def run_inference(model, tokenizer, df, batch_size=128):
    model.eval()
    preds = []
    sents = df["sentence"].tolist()
    with torch.inference_mode():
        for i in range(0, len(sents), batch_size):
            enc = tokenizer(sents[i:i+batch_size], truncation=True, padding=True,
                            max_length=128, return_tensors="pt").to(model.device)
            logits = model(**enc).logits
            preds.extend(ID2LABEL[p] for p in logits.argmax(-1).tolist())
    return preds

def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x if x in LABEL_SET else "neutral" for x in pred_labels]
    return {
        "macro_f1":    f1_score(true_labels, pred_labels, labels=LABELS, average="macro",    zero_division=0),
        "weighted_f1": f1_score(true_labels, pred_labels, labels=LABELS, average="weighted", zero_division=0),
        "accuracy":    accuracy_score(true_labels, pred_labels),
        "mcc":         matthews_corrcoef(true_labels, pred_labels),
        "f1_negative": f1_score(true_labels, pred_labels, labels=["negative"], average="macro", zero_division=0),
        "f1_neutral":  f1_score(true_labels, pred_labels, labels=["neutral"],  average="macro", zero_division=0),
        "f1_positive": f1_score(true_labels, pred_labels, labels=["positive"], average="macro", zero_division=0),
        "report":      classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0),
    }

def print_metrics(title, m):
    print("="*70, title, "="*70, m["report"], sep="\n")
    for k in ("macro_f1","weighted_f1","accuracy","mcc","f1_negative","f1_neutral","f1_positive"):
        print(f"  {k}: {m[k]:.4f}")

def free_gpu(*objs):
    for o in objs: del o
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## Stage 1 — Baseline (pretrained FinBERT on FPB test set)

In [ ]:
base_tok   = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)

# FinBERT uses its own label ordering; remap to our standard
finbert_id2label = {v.lower(): k for k, v in base_model.config.id2label.items()}
def remap_finbert_pred(p):
    label = base_model.config.id2label.get(p, "neutral").lower()
    return label if label in LABEL_SET else "neutral"

base_preds_raw = []
sents = test_df["sentence"].tolist()
with torch.inference_mode():
    for i in range(0, len(sents), 128):
        enc = base_tok(sents[i:i+128], truncation=True, padding=True, max_length=128, return_tensors="pt").to(DEVICE)
        logits = base_model(**enc).logits
        base_preds_raw.extend(logits.argmax(-1).tolist())

base_preds   = [remap_finbert_pred(p) for p in base_preds_raw]
base_metrics = evaluate_predictions(test_df["label_text"].tolist(), base_preds)
print_metrics("FinBERT — Pretrained baseline on FPB test", base_metrics)
free_gpu(base_model)

## Stage 2 — LoRA Fine-tuning + Random Search (val macro-F1 via compute_metrics)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

SEARCH_SPACE = {
    "lora_r":        [8, 16],
    "lora_alpha":    [16, 32],
    "lora_dropout":  [0.0, 0.05],
    "learning_rate": [2e-5, 5e-5],
    "batch_size":    [16, 32],
    "epochs":        [3, 5],
}
rng = random.Random(SEED)
sample_params = lambda: {k: rng.choice(v) for k, v in SEARCH_SPACE.items()}

best_val_f1 = -1.0; best_params = None; best_trial = -1; trial_log = []

for trial in range(RANDOM_SEARCH_TRIALS):
    params = sample_params()
    print(f"\n{'='*60}\nTrial {trial+1}/{RANDOM_SEARCH_TRIALS}: {params}\n{'='*60}")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=params["lora_r"], lora_alpha=params["lora_alpha"], lora_dropout=params["lora_dropout"],
        target_modules=["query","value"],
    )
    model = get_peft_model(model, lora_cfg)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    trial_ckpt = CKPT_BASE / f"trial_{trial}"

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(trial_ckpt),
            num_train_epochs=params["epochs"],
            per_device_train_batch_size=params["batch_size"],
            per_device_eval_batch_size=params["batch_size"]*2,
            learning_rate=params["learning_rate"],
            lr_scheduler_type="cosine", warmup_ratio=0.05,
            weight_decay=0.01,
            eval_strategy="epoch", save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1", greater_is_better=True,
            fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            dataloader_num_workers=4, seed=SEED, report_to="none",
            logging_steps=20,
        ),
        train_dataset=FPBDataset(train_df, tokenizer),
        eval_dataset=FPBDataset(val_df, tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    val_preds   = run_inference(model, tokenizer, val_df)
    val_metrics = evaluate_predictions(val_df["label_text"].tolist(), val_preds)
    val_f1      = val_metrics["macro_f1"]
    print(f"Trial {trial+1} | val macro-F1: {val_f1:.4f}")
    trial_log.append({"trial": trial+1, "val_macro_f1": val_f1, "params": params})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1; best_params = params; best_trial = trial+1
        model.save_pretrained(str(CKPT_BASE/"best_adapter"))
        tokenizer.save_pretrained(str(CKPT_BASE/"best_adapter"))
        print(f"  ↑ New best saved (trial {best_trial}, val macro-F1={best_val_f1:.4f})")

    del trainer, model, tokenizer; free_gpu()

if best_params is None: raise RuntimeError("All trials failed.")
print(f"\nBest trial {best_trial} | val macro-F1 {best_val_f1:.4f} | params {best_params}")

In [ ]:
# ─── Retrain on train+val with best params ────────────────────────────────────
FINAL_CKPT = CKPT_BASE / "final_best"

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True,
)
final_model = get_peft_model(final_model, LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=best_params["lora_r"], lora_alpha=best_params["lora_alpha"], lora_dropout=best_params["lora_dropout"],
    target_modules=["query","value"],
))
final_tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

final_trainer = Trainer(
    model=final_model,
    args=TrainingArguments(
        output_dir=str(FINAL_CKPT),
        num_train_epochs=best_params["epochs"],
        per_device_train_batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"],
        lr_scheduler_type="cosine", warmup_ratio=0.05,
        weight_decay=0.01,
        save_strategy="epoch", logging_steps=20,
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        dataloader_num_workers=4, seed=SEED, report_to="none",
    ),
    train_dataset=FPBDataset(train_val_df, final_tok),
    compute_metrics=compute_metrics,
)
final_trainer.train()
del final_trainer; gc.collect()

# Save merged model (LoRA weights merged into base)
MERGED_DIR = CKPT_BASE / "merged_model"
merged_model = final_model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
final_tok.save_pretrained(str(MERGED_DIR))
print(f"Merged model saved → {MERGED_DIR}")

## Stage 3 — Final Evaluation on Test Set (best hyperparams from Stage 2)

In [ ]:
test_preds    = run_inference(merged_model, final_tok, test_df)
final_metrics = evaluate_predictions(test_df["label_text"].tolist(), test_preds)
print_metrics(f"FinBERT LoRA (Stage 2 best trial {best_trial}) — FPB TEST SET", final_metrics)

(CKPT_BASE / "inference_config.json").write_text(json.dumps({
    "model_type": "sequence_classification",
    "model_name_or_path": str(MERGED_DIR),
    "base_model": MODEL_NAME,
    "adapter_path": str(CKPT_BASE / "best_adapter"),
    "labels": LABELS,
    "label2id": LABEL2ID,
    "id2label": {str(k): v for k, v in ID2LABEL.items()},
    "val_macro_f1": round(best_val_f1, 4),
    "test_macro_f1": round(final_metrics["macro_f1"], 4),
    "best_trial": best_trial,
    "best_params": best_params,
}, indent=2), encoding="utf-8")
print(f"inference_config.json → {CKPT_BASE}")

In [ ]:
rows = [
    {"stage": "Baseline (pretrained FinBERT)",      **{k: v for k, v in base_metrics.items()  if isinstance(v, float)}},
    {"stage": f"LoRA best (trial {best_trial})",   **{k: v for k, v in final_metrics.items() if isinstance(v, float)}},
]
cols = ["stage","macro_f1","weighted_f1","accuracy","mcc","f1_negative","f1_neutral","f1_positive"]
display(pd.DataFrame(rows)[cols].sort_values("macro_f1", ascending=False).reset_index(drop=True))
print("\nTrial log:", *[f"  {r}" for r in trial_log], sep="\n")

In [ ]:
for v in ["merged_model","final_model","final_tok","base_model","base_tok"]:
    if v in globals(): del globals()[v]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Done.")